# AnimalCLEF2026 LB 0.32903 Lynx ALIKED Merge-Only Reproducer

This notebook reproduces the current best submitted public-LB result:

```text
0.32903 = 0.32684 final Texas boundary split-only base
        + six ultra-strict Lynx EVA02/ALIKED cross-cluster merges
```

It does **not** load the submitted `0.32903` CSV. It recomputes the final labels from:

1. the verified `0.32684` base output from `final-texas-boundary-split-only`
2. the Lynx heavy notebook pair-score report from `lynx-low-light-eva-clip-aliked`

## Required Kaggle Inputs

Attach:

- `animal-clef-2026`
- output of `hanifnoerrofiq/final-texas-boundary-split-only`
- output of `hanifnoerrofiq/lynx-low-light-eva-clip-aliked`

This notebook is CPU-only because the expensive EVA02 + ALIKED pair scoring has already been produced by the Lynx notebook.

## Output To Submit

Use:

```text
/kaggle/working/animalclef_lb032903_lynx_aliked_reproducer_v20260430/submission.csv
```
            

In [ ]:
from pathlib import Path

SCRIPT_TEXT = "#!/usr/bin/env python3\n\"\"\"\nAnimalCLEF2026 LB 0.32903 Lynx ALIKED merge-only reproducer.\n\nThis notebook/script reproduces the submitted 0.32903 public-LB artifact\nwithout loading the submitted CSV. It starts from the verified 0.32684\nfinal-texas-boundary-split-only partition, reads the Lynx EVA02+ALIKED\ntest pair scores from the heavy Lynx notebook output, and applies the exact\nultra-strict merge-only rule that improved the leaderboard.\n\nRule:\n    cross-current-cluster Lynx pair\n    not opposite flank\n    local_score >= 0.96\n    inliers >= 50\n    match_prob >= 0.98\n    merged component size <= 84\n\nOnly Lynx labels change. Salamander, SeaTurtle, and Texas remain identical to\nthe 0.32684 base partition.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport itertools\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport pandas as pd\n\n\nVERSION = \"lynx_aliked_best_reproducer_v20260430\"\nLYNX = \"LynxID2025\"\n\nBASE_CANDIDATES = [\n    \"submission_final_texas_boundary_splitonly_balanced_from_032583_v20260430.csv\",\n    \"submission.csv\",\n]\nPAIR_SCORE_NAME = \"lynx_lowlight_eva_aliked_v20260430_test_pair_scores.csv\"\nSUBMITTED_NAME = \"submission_032684_lynx_aliked_prob098_mergeonly_v20260430.csv\"\nSCORE_NAME = \"submission_lb032903_lynx_aliked_prob098_mergeonly_v20260430.csv\"\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--base-submission\", type=str, default=None)\n    parser.add_argument(\"--pair-scores\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=f\"/kaggle/working/animalclef_{VERSION}\")\n    parser.add_argument(\"--reference-submission\", type=str, default=None)\n    return parser.parse_args()\n\n\ndef candidate_roots() -> list[Path]:\n    roots = []\n    for value in [\n        os.environ.get(\"DATA_ROOT\"),\n        \"/kaggle/input/competitions/animal-clef-2026\",\n        \"/kaggle/input/animal-clef-2026\",\n        \"/kaggle/input\",\n        \"/kaggle/working\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026/animal-clef-2026\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026/current_wildfusion_graph_v20260423\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026/kaggle_output_lynx_lowlight_eva_aliked_v315501274\",\n        \"C:/Users/Hanif/Documents/kaggle/AnimalCLEF2026\",\n        \".\",\n    ]:\n        if value:\n            p = Path(value)\n            if p.exists():\n                roots.append(p)\n    out = []\n    seen = set()\n    for root in roots:\n        try:\n            key = str(root.resolve())\n        except Exception:\n            key = str(root)\n        if key not in seen:\n            out.append(root)\n            seen.add(key)\n    return out\n\n\ndef find_data_root(arg: str | None) -> Path:\n    if arg:\n        p = Path(arg)\n        if (p / \"metadata.csv\").exists() and (p / \"sample_submission.csv\").exists():\n            return p.resolve()\n        raise FileNotFoundError(f\"--data-root is not an AnimalCLEF root: {p}\")\n    for root in candidate_roots():\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    for root in candidate_roots():\n        try:\n            for meta in root.rglob(\"metadata.csv\"):\n                if (meta.parent / \"sample_submission.csv\").exists():\n                    return meta.parent.resolve()\n        except Exception:\n            pass\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef rank_base_path(path: Path) -> tuple[int, int, str]:\n    text = str(path).replace(\"\\\\\", \"/\").lower()\n    penalty = 0\n    if \"local_smoke\" in text or \"__pycache__\" in text:\n        penalty += 100\n    if \"final-texas-boundary-split-only\" in text:\n        penalty -= 50\n    if \"animalclef_texas_boundary_splitonly_final\" in text:\n        penalty -= 40\n    if \"submission_final_texas_boundary_splitonly\" in path.name.lower():\n        penalty -= 30\n    if \"/kaggle/input/\" in text:\n        penalty -= 10\n    if path.name == \"submission.csv\":\n        penalty += 8\n    return penalty, len(text), text\n\n\ndef rank_pair_path(path: Path) -> tuple[int, int, str]:\n    text = str(path).replace(\"\\\\\", \"/\").lower()\n    penalty = 0\n    if \"local_smoke\" in text or \"__pycache__\" in text:\n        penalty += 100\n    if \"lynx-low-light-eva-clip-aliked\" in text:\n        penalty -= 50\n    if \"animalclef_lynx_lowlight_eva_aliked\" in text:\n        penalty -= 40\n    if \"/reports/\" in text:\n        penalty -= 10\n    if \"/kaggle/input/\" in text:\n        penalty -= 10\n    return penalty, len(text), text\n\n\ndef find_file(names: Iterable[str], arg: str | None, ranker) -> Path:\n    if arg:\n        p = Path(arg)\n        if p.exists():\n            return p.resolve()\n        raise FileNotFoundError(f\"Path does not exist: {p}\")\n    hits = []\n    for root in candidate_roots():\n        for name in names:\n            direct = root / name\n            if direct.exists():\n                hits.append(direct)\n            try:\n                hits.extend(root.rglob(name))\n            except Exception:\n                pass\n    hits = sorted({p.resolve() for p in hits if p.exists()}, key=ranker)\n    if not hits:\n        raise FileNotFoundError(f\"Could not find any of: {list(names)}\")\n    return hits[0]\n\n\nclass UF:\n    def __init__(self, values: Iterable[int]):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> bool:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return False\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return True\n\n\ndef load_submission(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    if \"image_id\" not in df.columns or \"cluster\" not in df.columns:\n        raise ValueError(f\"{path} must contain image_id and cluster columns.\")\n    df = df[[\"image_id\", \"cluster\"]].copy()\n    df[\"image_id\"] = df[\"image_id\"].astype(int)\n    df[\"cluster\"] = df[\"cluster\"].astype(str)\n    if df[\"image_id\"].duplicated().any():\n        raise ValueError(f\"Duplicate image_id values in {path}\")\n    return df\n\n\ndef validate_submission(sub: pd.DataFrame, sample: pd.DataFrame) -> None:\n    if list(sub.columns) != [\"image_id\", \"cluster\"]:\n        raise ValueError(\"Submission columns must be exactly image_id, cluster.\")\n    if len(sub) != len(sample):\n        raise ValueError(f\"Submission rows {len(sub)} != sample rows {len(sample)}\")\n    if sub[\"image_id\"].astype(int).tolist() != sample[\"image_id\"].astype(int).tolist():\n        raise ValueError(\"Submission image order does not match sample_submission.csv\")\n    if sub[\"cluster\"].isna().any():\n        raise ValueError(\"Submission contains null cluster labels.\")\n    max_len = int(sub[\"cluster\"].astype(str).str.len().max())\n    if max_len > 64:\n        raise ValueError(f\"Cluster labels too long: max length {max_len}\")\n\n\ndef compact_species_labels(sub: pd.DataFrame, sample: pd.DataFrame, test_rows: pd.DataFrame) -> pd.DataFrame:\n    parts = []\n    sample_ids = set(sample[\"image_id\"].astype(int))\n    for species, group in test_rows[test_rows[\"image_id\"].isin(sample_ids)].groupby(\"dataset\", sort=True):\n        ids = set(group[\"image_id\"].astype(int))\n        part = sub[sub[\"image_id\"].astype(int).isin(ids)].copy()\n        groups = []\n        for _, members in part.groupby(\"cluster\"):\n            groups.append(sorted(members[\"image_id\"].astype(int).tolist()))\n        labels = {}\n        for idx, members in enumerate(sorted(groups, key=lambda xs: (min(xs), len(xs), max(xs)))):\n            for image_id in members:\n                labels[image_id] = f\"cluster_{species}_{idx}\"\n        parts.append(\n            pd.DataFrame(\n                {\n                    \"image_id\": sorted(ids),\n                    \"cluster\": [labels[image_id] for image_id in sorted(ids)],\n                }\n            )\n        )\n    final = pd.concat(parts, ignore_index=True)\n    final = sample[[\"image_id\"]].merge(final, on=\"image_id\", how=\"left\")\n    return final[[\"image_id\", \"cluster\"]]\n\n\ndef shape_report(sub: pd.DataFrame, test_rows: pd.DataFrame) -> pd.DataFrame:\n    tmp = sub.merge(test_rows[[\"image_id\", \"dataset\"]], on=\"image_id\", how=\"left\")\n    rows = []\n    for species, group in tmp.groupby(\"dataset\", sort=True):\n        sizes = group.groupby(\"cluster\").size()\n        rows.append(\n            {\n                \"species\": species,\n                \"n_images\": int(len(group)),\n                \"n_clusters\": int(sizes.size),\n                \"max_cluster_size\": int(sizes.max()),\n                \"singletons\": int((sizes == 1).sum()),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef file_sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open(\"rb\") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef build_best_submission(base: pd.DataFrame, sample: pd.DataFrame, metadata: pd.DataFrame, pair_scores: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    test_rows = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    lynx_ids = (\n        test_rows[test_rows[\"dataset\"].eq(LYNX) & test_rows[\"image_id\"].isin(sample[\"image_id\"])]\n        [\"image_id\"]\n        .astype(int)\n        .tolist()\n    )\n    current = base[base[\"image_id\"].astype(int).isin(lynx_ids)].copy()\n\n    uf = UF(lynx_ids)\n    for _, group in current.groupby(\"cluster\"):\n        ids = group[\"image_id\"].astype(int).tolist()\n        for a, b in itertools.combinations(ids, 2):\n            uf.union(a, b)\n\n    if pair_scores[\"same_current_cluster\"].dtype != bool:\n        same_current = pair_scores[\"same_current_cluster\"].astype(str).str.lower().isin([\"true\", \"1\", \"yes\"])\n    else:\n        same_current = pair_scores[\"same_current_cluster\"]\n\n    accepted = pair_scores[\n        (~same_current)\n        & (~pair_scores[\"orientation_relation\"].astype(str).eq(\"opposite_flank\"))\n        & (pair_scores[\"local_score\"].astype(float) >= 0.96)\n        & (pair_scores[\"inliers\"].astype(float) >= 50)\n        & (pair_scores[\"match_prob\"].astype(float) >= 0.98)\n    ].copy()\n    accepted = accepted.sort_values([\"local_score\", \"inliers\", \"match_prob\"], ascending=False)\n\n    actions = []\n    for row in accepted.itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        if a not in uf.parent or b not in uf.parent:\n            continue\n        if uf.find(a) == uf.find(b):\n            continue\n        size_after = uf.size[uf.find(a)] + uf.size[uf.find(b)]\n        if size_after > 84:\n            continue\n        uf.union(a, b)\n        actions.append(\n            {\n                \"image_id_a\": a,\n                \"image_id_b\": b,\n                \"cluster_a\": row.cluster_a,\n                \"cluster_b\": row.cluster_b,\n                \"match_prob\": float(row.match_prob),\n                \"eva_sim01\": float(row.eva_sim01),\n                \"local_score\": float(row.local_score),\n                \"matches\": int(row.matches),\n                \"inliers\": int(row.inliers),\n                \"orientation_relation\": row.orientation_relation,\n                \"size_after\": int(size_after),\n            }\n        )\n\n    groups = {}\n    for image_id in lynx_ids:\n        groups.setdefault(uf.find(image_id), []).append(image_id)\n    labels = {}\n    for idx, members in enumerate(sorted(groups.values(), key=lambda xs: (min(xs), len(xs), max(xs)))):\n        for image_id in members:\n            labels[image_id] = f\"cluster_{LYNX}_{idx}\"\n\n    sub = base.copy()\n    mask = sub[\"image_id\"].astype(int).isin(labels)\n    sub.loc[mask, \"cluster\"] = sub.loc[mask, \"image_id\"].astype(int).map(labels)\n    final = compact_species_labels(sub, sample, test_rows)\n    return final, pd.DataFrame(actions)\n\n\ndef main() -> None:\n    args = parse_args()\n    data_root = find_data_root(args.data_root)\n    base_path = find_file(BASE_CANDIDATES, args.base_submission, rank_base_path)\n    pair_path = find_file([PAIR_SCORE_NAME], args.pair_scores, rank_pair_path)\n\n    output_root = Path(args.output_root)\n    reports_dir = output_root / \"reports\"\n    submissions_dir = output_root / \"submissions\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    submissions_dir.mkdir(parents=True, exist_ok=True)\n\n    metadata = pd.read_csv(data_root / \"metadata.csv\").copy()\n    if \"dataset\" not in metadata.columns:\n        metadata[\"dataset\"] = metadata[\"path\"].astype(str).str.replace(\"\\\\\", \"/\", regex=False).str.split(\"/\").str[1]\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = metadata[\"path\"].astype(str).str.replace(\"\\\\\", \"/\", regex=False).str.contains(\"/test/\").map({True: \"test\", False: \"train\"})\n    metadata[\"image_id\"] = metadata[\"image_id\"].astype(int)\n    test_rows = metadata[metadata[\"split\"].eq(\"test\")].copy()\n\n    sample = pd.read_csv(data_root / \"sample_submission.csv\")[[\"image_id\"]].copy()\n    sample[\"image_id\"] = sample[\"image_id\"].astype(int)\n    base = load_submission(base_path)\n    base_for_validation = sample.merge(base, on=\"image_id\", how=\"left\")\n    validate_submission(base_for_validation, sample)\n    pair_scores = pd.read_csv(pair_path)\n\n    final, actions = build_best_submission(base_for_validation, sample, metadata, pair_scores)\n    validate_submission(final, sample)\n\n    top_level = output_root / \"submission.csv\"\n    submitted_name_path = submissions_dir / SUBMITTED_NAME\n    score_name_path = submissions_dir / SCORE_NAME\n    final.to_csv(top_level, index=False)\n    final.to_csv(submitted_name_path, index=False)\n    final.to_csv(score_name_path, index=False)\n\n    actions_path = reports_dir / f\"{VERSION}_actions.csv\"\n    shape_path = reports_dir / f\"{VERSION}_shape_report.csv\"\n    actions.to_csv(actions_path, index=False)\n    shape = shape_report(final, test_rows)\n    shape.to_csv(shape_path, index=False)\n\n    base_shape = shape_report(base_for_validation, test_rows)\n    base_shape.to_csv(reports_dir / f\"{VERSION}_base_shape_report.csv\", index=False)\n\n    reference_match = None\n    reference_sha = None\n    if args.reference_submission:\n        ref = Path(args.reference_submission)\n        if ref.exists():\n            ref_df = pd.read_csv(ref)[[\"image_id\", \"cluster\"]]\n            ref_df[\"image_id\"] = ref_df[\"image_id\"].astype(int)\n            ref_df[\"cluster\"] = ref_df[\"cluster\"].astype(str)\n            reference_match = bool(final.equals(ref_df))\n            reference_sha = file_sha256(ref)\n\n    summary = {\n        \"version\": VERSION,\n        \"public_lb\": \"0.32903\",\n        \"base_public_lb\": \"0.32684\",\n        \"data_root\": str(data_root),\n        \"base_submission\": str(base_path),\n        \"pair_scores\": str(pair_path),\n        \"top_level_submission\": str(top_level),\n        \"submitted_name_copy\": str(submitted_name_path),\n        \"score_name_copy\": str(score_name_path),\n        \"n_actions\": int(len(actions)),\n        \"actions\": actions.to_dict(\"records\"),\n        \"output_sha256\": file_sha256(top_level),\n        \"reference_sha256\": reference_sha,\n        \"matches_reference_submission\": reference_match,\n        \"notes\": [\n            \"This reproduces the 0.32903 artifact by recomputing labels from the 0.32684 base and the Lynx EVA02+ALIKED pair-score report.\",\n            \"It does not load the submitted 0.32903 CSV unless --reference-submission is passed for validation.\",\n            \"Only six Lynx cross-cluster merges are accepted; all non-Lynx labels are inherited from the 0.32684 base.\",\n        ],\n    }\n    summary_path = reports_dir / f\"{VERSION}_summary.json\"\n    summary_path.write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n\n    print(\"Base shape:\")\n    print(base_shape.to_string(index=False))\n    print(\"\\nFinal shape:\")\n    print(shape.to_string(index=False))\n    print(\"\\nAccepted Lynx merges:\")\n    print(actions.to_string(index=False))\n    print(\"\\nSummary:\")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n"
Path("lynx_aliked_best_reproducer_v20260430.py").write_text(SCRIPT_TEXT, encoding="utf-8")
print("wrote lynx_aliked_best_reproducer_v20260430.py")
            

In [ ]:
from pathlib import Path
import subprocess
import sys

OUTPUT_ROOT = Path("/kaggle/working/animalclef_lb032903_lynx_aliked_reproducer_v20260430")

# Usually keep these as None. The script prefers the correct Kaggle output
# folders automatically. If you attach many old notebooks with ambiguous
# submission.csv files, set these manually.
BASE_SUBMISSION = None
PAIR_SCORES = None

cmd = [
    sys.executable,
    "lynx_aliked_best_reproducer_v20260430.py",
    "--output-root", str(OUTPUT_ROOT),
]
if BASE_SUBMISSION:
    cmd.extend(["--base-submission", BASE_SUBMISSION])
if PAIR_SCORES:
    cmd.extend(["--pair-scores", PAIR_SCORES])

print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)
            

In [ ]:
from pathlib import Path
import json
import pandas as pd

out = Path("/kaggle/working/animalclef_lb032903_lynx_aliked_reproducer_v20260430")
reports = out / "reports"
summary = json.loads((reports / "lynx_aliked_best_reproducer_v20260430_summary.json").read_text())

print(json.dumps(summary, indent=2))
display(pd.read_csv(reports / "lynx_aliked_best_reproducer_v20260430_base_shape_report.csv"))
display(pd.read_csv(reports / "lynx_aliked_best_reproducer_v20260430_shape_report.csv"))
display(pd.read_csv(reports / "lynx_aliked_best_reproducer_v20260430_actions.csv"))

submission = pd.read_csv(out / "submission.csv")
print("Submit:", out / "submission.csv")
print("Rows:", len(submission))
print("Max label length:", submission["cluster"].astype(str).str.len().max())
print("Expected SHA256 from local submitted artifact:")
print("ab0a4c1fef7ce3a049d2c71b1999da52f6b4463a01649849d90cc834f7bda449")
print("Produced SHA256:")
print(summary["output_sha256"])
            